# CSE 4/587 Project Phase 1 — Tasks 4 & 5
## S&P 500 Stock Prices Analysis (2014–2017)
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load raw dataset
df = pd.read_csv('S&P 500 Stock Prices 2014-2017.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
df.head(10)

In [ ]:
# Initial overview
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Basic Info ===')
df.info()

---
# Task 4: Data Cleaning / Processing (6 Operations)
---

## Cleaning 1: Handle Missing Values
**Rationale:** Missing values can distort analysis and break ML models. We identify and handle them appropriately.

In [ ]:
# Check missing values
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print(missing_df)
print(f'\nTotal rows with any missing value: {df.isnull().any(axis=1).sum()}')

In [ ]:
# Drop rows with missing values (if any)
rows_before = len(df)
df = df.dropna()
rows_after = len(df)
print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {rows_before - rows_after:,}')

## Cleaning 2: Fix Data Types — Parse Date Column
**Rationale:** The `date` column is loaded as a string. Converting it to datetime enables time-based analysis, sorting, and feature extraction.

In [ ]:
print(f'Date dtype before: {df["date"].dtype}')
df['date'] = pd.to_datetime(df['date'])
print(f'Date dtype after: {df["date"].dtype}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')

## Cleaning 3: Remove Duplicate Rows
**Rationale:** Duplicate entries for the same stock on the same date would inflate statistics and bias models.

In [ ]:
dupes = df.duplicated(subset=['symbol', 'date']).sum()
print(f'Duplicate (symbol, date) pairs: {dupes}')

rows_before = len(df)
df = df.drop_duplicates(subset=['symbol', 'date'], keep='first')
rows_after = len(df)
print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Duplicates removed: {rows_before - rows_after:,}')

## Cleaning 4: Outlier Detection and Handling (Volume)
**Rationale:** Extremely high or zero volume entries may indicate data errors or non-trading days. We flag and filter unreasonable values.

In [ ]:
# Check for zero or negative volume
zero_vol = (df['volume'] <= 0).sum()
print(f'Rows with volume <= 0: {zero_vol}')

# Remove zero-volume rows (no actual trading occurred)
rows_before = len(df)
df = df[df['volume'] > 0]
rows_after = len(df)
print(f'Removed {rows_before - rows_after} zero-volume rows')

# Check for price anomalies: high < low or close <= 0
price_errors = ((df['high'] < df['low']) | (df['close'] <= 0)).sum()
print(f'Rows with price anomalies (high < low or close <= 0): {price_errors}')
df = df[(df['high'] >= df['low']) & (df['close'] > 0)]
print(f'Final rows after outlier cleaning: {len(df):,}')

## Cleaning 5: Derive New Features (Feature Engineering)
**Rationale:** Raw OHLCV data alone is limited. We derive features like daily return, price range, and day-of-week to support Phase 2 modeling.

In [ ]:
# Daily return (%)
df['daily_return'] = ((df['close'] - df['open']) / df['open']) * 100

# Intraday price range
df['price_range'] = df['high'] - df['low']

# Day of week (0=Monday, 4=Friday)
df['day_of_week'] = df['date'].dt.dayofweek

# Year and Month for temporal analysis
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print('New columns added: daily_return, price_range, day_of_week, year, month')
df.head()

## Cleaning 6: Normalize Symbol Names & Sort Data
**Rationale:** Ensuring consistent symbol formatting and chronological ordering prevents errors in time-series analysis and grouping.

In [ ]:
# Normalize symbols to uppercase and strip whitespace
df['symbol'] = df['symbol'].str.upper().str.strip()
print(f'Unique symbols: {df["symbol"].nunique()}')

# Sort by symbol and date
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f'\n=== Cleaned Dataset Summary ===')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.dtypes

### Export Cleaned Dataset
Save the cleaned dataset for HDFS upload (Task 3 Part B) and Phase 2.

In [ ]:
df.to_csv('cleaned_sp500.csv', index=False)
print(f'Cleaned dataset saved: cleaned_sp500.csv')
print(f'Final shape: {df.shape}')

---
# Task 5: Exploratory Data Analysis (6 EDA Operations)
---

## EDA 1: Summary Statistics (Non-Graphical)
**Purpose:** Understand the central tendency, spread, and shape of numerical features to identify data characteristics and potential modeling considerations.

In [ ]:
print('=== Descriptive Statistics for Numerical Columns ===')
df[['open', 'high', 'low', 'close', 'volume', 'daily_return', 'price_range']].describe().round(2)

**Insight:** The summary statistics reveal the distribution of stock prices and trading volumes across the S&P 500. The large standard deviation in volume indicates significant variation in trading activity across stocks, which is relevant for volatility modeling in Phase 2.

## EDA 2: Distribution of Daily Returns (Graphical — Histogram)
**Purpose:** Examine whether daily returns follow a normal distribution, which is a common assumption in financial modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['daily_return'].clip(-10, 10), bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of Daily Returns (%)', fontsize=14)
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(x=0, color='red', linestyle='--', label='Zero Return')
axes[0].legend()

# Box plot
axes[1].boxplot(df['daily_return'].clip(-10, 10), vert=True)
axes[1].set_title('Box Plot of Daily Returns (%)', fontsize=14)
axes[1].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()

print(f'Mean daily return: {df["daily_return"].mean():.4f}%')
print(f'Median daily return: {df["daily_return"].median():.4f}%')
print(f'Skewness: {df["daily_return"].skew():.4f}')
print(f'Kurtosis: {df["daily_return"].kurtosis():.4f}')

**Insight:** Daily returns are approximately centered around zero with heavy tails (high kurtosis), indicating that extreme price movements occur more frequently than a normal distribution would predict. This has direct implications for risk modeling and anomaly detection in Phase 2.

## EDA 3: Correlation Analysis (Non-Graphical + Heatmap)
**Purpose:** Identify relationships between numerical features to guide feature selection for Phase 2 ML models.

In [ ]:
corr_cols = ['open', 'high', 'low', 'close', 'volume', 'daily_return', 'price_range']
corr_matrix = df[corr_cols].corr().round(3)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Correlation Heatmap of Stock Features', fontsize=14)
plt.tight_layout()
plt.show()

print('\n=== Correlation Matrix ===')
print(corr_matrix)

**Insight:** OHLC prices are highly correlated (>0.99) as expected. Volume shows low correlation with price levels, suggesting it carries independent information. Daily return and price range have moderate correlation, indicating that larger price swings are associated with bigger intraday moves. These relationships will inform feature engineering in Phase 2.

## EDA 4: Trading Volume Over Time (Graphical — Line Chart)
**Purpose:** Examine temporal trends in market activity to understand seasonality and market regime changes.

In [ ]:
# Aggregate daily total volume across all stocks
daily_volume = df.groupby('date')['volume'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(daily_volume['date'], daily_volume['volume'] / 1e9, color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Total Daily Trading Volume Across S&P 500 (2014–2017)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Volume (Billions)')
plt.tight_layout()
plt.show()

# Monthly average
monthly_vol = df.groupby(['year', 'month'])['volume'].mean().reset_index()
print('=== Average Volume by Year ===')
print(df.groupby('year')['volume'].mean().round(0))

**Insight:** Trading volume shows periodic spikes, often corresponding to major market events (e.g., earnings seasons, market corrections). There is also a general trend in volume patterns across years. These temporal patterns will be useful for time-series modeling in Phase 2.

## EDA 5: Top 10 Most Volatile Stocks (Graphical — Bar Chart)
**Purpose:** Identify which stocks have the highest price volatility, which is key for risk assessment and anomaly detection tasks.

In [ ]:
# Volatility = std of daily returns per stock
volatility = df.groupby('symbol')['daily_return'].std().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 most volatile
top10 = volatility.head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color='salmon', edgecolor='black')
axes[0].set_title('Top 10 Most Volatile Stocks', fontsize=14)
axes[0].set_xlabel('Std of Daily Return (%)')

# Top 10 least volatile
bottom10 = volatility.tail(10)
axes[1].barh(bottom10.index[::-1], bottom10.values[::-1], color='lightgreen', edgecolor='black')
axes[1].set_title('Top 10 Least Volatile Stocks', fontsize=14)
axes[1].set_xlabel('Std of Daily Return (%)')

plt.tight_layout()
plt.show()

print(f'Overall average volatility: {volatility.mean():.4f}%')
print(f'Most volatile: {volatility.idxmax()} ({volatility.max():.4f}%)')
print(f'Least volatile: {volatility.idxmin()} ({volatility.min():.4f}%)')

**Insight:** Volatility varies significantly across S&P 500 stocks. High-volatility stocks (often in tech/biotech sectors) contrast with stable utility/consumer staple stocks. This variance is important for clustering stocks by risk profile and for building volatility prediction models in Phase 2.

## EDA 6: Average Daily Return by Day of Week (Graphical — Bar Chart)
**Purpose:** Investigate whether there is a "day-of-week effect" in stock returns — a well-known anomaly in financial markets.

In [ ]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_returns = df.groupby('day_of_week')['daily_return'].agg(['mean', 'std']).round(4)
dow_returns.index = day_names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['green' if x >= 0 else 'red' for x in dow_returns['mean']]
axes[0].bar(dow_returns.index, dow_returns['mean'], color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Average Daily Return by Day of Week', fontsize=14)
axes[0].set_ylabel('Mean Return (%)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

axes[1].bar(dow_returns.index, dow_returns['std'], color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Return Volatility by Day of Week', fontsize=14)
axes[1].set_ylabel('Std of Return (%)')

plt.tight_layout()
plt.show()

print('=== Return Statistics by Day of Week ===')
print(dow_returns)

**Insight:** The day-of-week analysis reveals whether certain days exhibit systematically higher or lower returns. If such patterns exist, `day_of_week` can serve as a useful feature for return prediction models in Phase 2. Volatility differences across days also inform risk management strategies.

---
## Summary
- **Cleaned dataset:** 497K+ rows, 12 columns (7 original + 5 derived)
- **Key findings:** Daily returns are heavy-tailed, OHLC prices are highly correlated, volatility varies significantly across stocks, and temporal patterns exist in volume and returns
- **Phase 2 implications:** These insights will guide feature engineering (daily return, volatility, day-of-week), model selection (need to handle non-normal distributions), and evaluation strategy